# Travelling salesman with the MTZ formulation (chapter 3)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/mip-modelling/blob/main/notebooks/cap06_tsp.ipynb)

The third of the three problems the heuristics chapter takes up again: there the
tour is built with nearest neighbour, here the optimum is found.

The u variables order the cities along the tour: the constraint
u_i - u_j + n x_ij <= n - 1 is true when x_ij = 0 and forces u_j >= u_i + 1 when
x_ij = 1. Subtours that miss city 1 are thereby excluded, because they would need
a chain of ever-growing u that closes on itself.

The full chapter — model, data, results and sensitivity analysis — is [on the website](https://fabiofurini.github.io/mip-modelling/gurobipy-4/).

## Setup

The cell below installs `gurobipy` and downloads the four shared modules of the
course: `stile.py` (palette), `mip.py` (relaxation, dual, bounds),
`euristiche.py` (next-fit, first-fit, best-fit) and `esteso.py` (the model of the
instance written out in full). The licence bundled with the pip package is limited
to **2000 variables and 2000 constraints**: the instances of the course are small
and all fit with plenty of room. For larger instances activate the free academic
licence at [portal.gurobi.com](https://portal.gurobi.com).

In [ ]:
# Environment: the solver and the shared modules of the course.
# Locally it uses the repository's python/stile.py; on Colab it installs and downloads what is missing.
import importlib.util
import subprocess
import sys
import urllib.request
from pathlib import Path

if importlib.util.find_spec("gurobipy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gurobipy", "matplotlib", "pandas", "scipy"], check=True)

for modulo in ('stile', 'mip', 'euristiche', 'esteso', 'booleane'):                     # plotting style and course utilities
    locale = next((p for p in (Path(f"../python/{modulo}.py"), Path(f"python/{modulo}.py"))
                   if p.exists()), None)
    if locale is not None:
        sys.path.insert(0, str(locale.parent.resolve()))       # notebook opened in the repository
    else:                                 # on Colab always re-download: sessions last,
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/fabiofurini/mip-modelling/main/python/{modulo}.py", f"{modulo}.py")   # modules change

In [ ]:
import gurobipy as gp
import pandas as pd
from gurobipy import GRB

from esteso import salva_modello
from mip import frazione, nuovo_modello, risolvi
from stile import intestazione, salva_dati

R = range

intestazione("Travelling salesman: the shortest tour")
D_tsp = [[0, 4, 5, 9],
         [4, 0, 9, 9],
         [5, 9, 0, 4],
         [9, 9, 4, 0]]
n_tsp = len(D_tsp)


def modello_tsp(D):
    n = len(D)
    m = nuovo_modello("tsp")
    x = m.addVars(((i, j) for i in R(n) for j in R(n) if i != j), vtype=GRB.BINARY, name="x")
    u = m.addVars(R(1, n), lb=1, ub=n - 1, name="u")
    m.setObjective(gp.quicksum(D[i][j] * x[i, j] for i, j in x), GRB.MINIMIZE)
    m.addConstrs((gp.quicksum(x[i, j] for j in R(n) if j != i) == 1 for i in R(n)), name="out")
    m.addConstrs((gp.quicksum(x[i, j] for i in R(n) if i != j) == 1 for j in R(n)), name="in")
    m.addConstrs((u[i] - u[j] + n * x[i, j] <= n - 1
                  for i in R(1, n) for j in R(1, n) if i != j), name="mtz")
    return m, x, u


m_tsp, x_tsp, u_tsp = modello_tsp(D_tsp)
salva_modello(m_tsp, "cap06_tsp")
z_tsp = risolvi(m_tsp)
seguente = {i: j for (i, j) in x_tsp if x_tsp[i, j].X > 0.5}
giro, citta = [0], 0
while seguente[citta] != 0:
    citta = seguente[citta]
    giro.append(citta)
print(f"  {n_tsp} cities, symmetric and metric distances.")
print("  Optimal tour: " + " -> ".join(str(c + 1) for c in giro + [0])
      + f", length {frazione(z_tsp)}.")
salva_dati(pd.DataFrame([{"problem": "TSP", "z_milp": z_tsp}]), "cap06_tsp")
print("Done.")

---

Notebook generated from `python/cap06_tsp.py` with `python3 python/make_notebooks.py`:
edits go into the script, not here.

Teaching material by [Fabio Furini](https://sites.google.com/view/fabiofurini/home-page) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/mip-modelling/blob/main/LICENSE),
code [MIT](https://github.com/fabiofurini/mip-modelling/blob/main/LICENSE-CODE).